# Experimentul 10: BALANCED PCA vs FULL-TRAIN PCA pentru STAR ZZ

**Arhitectura fixă: STAR ZZ din Exp. 9** (4 qubiți, ZZ pe (0,1), (0,2), (0,3), feature map ×1, ansatz ×3, 32 de parametri, readout ⟨Z₀⟩, 15 CNOT, 73 de locuri de zgomot). Circuitul, zgomotul și antrenarea nu se schimbă.

Este o **ablație a encoderului clasic** care produce cele 4 unghiuri. Nu reduce porți, CNOT sau locuri de zgomot și nu este o metodă fizică de reducere a zgomotului.

| | Varianta A: BALANCED PCA (Exp. 1–9) | Varianta B: FULL-TRAIN PCA |
|---|---|---|
| `StandardScaler` (Time, Amount) | fit pe cele 590 de exemple 1:1 | **identic** (același obiect fitat pe cele 590) |
| `PCA(4).fit` | cele 590 de exemple 1:1 | **toate cele 170.883 din train** (standardizate cu scalerul de mai sus) |
| `MinMaxScaler(0, π).fit` | proiecțiile PCA-ului A ale celor 590 | proiecțiile PCA-ului B ale **acelorași 590** |
| clipping la [0, π] la transform | da | da (neschimbat) |
| VQC antrenat pe | cele 590 | cele 590 (VQC-ul nu vede niciodată tot train-ul) |

**H1:** PCA pe tot train-ul poate îmbunătăți recall-ul STAR păstrând AUPRC apropiat. **H0:** nicio îmbunătățire consistentă sau degradare, pentru că PCA pe date dezechilibrate urmărește variația tranzacțiilor legitime. Nu presupunem că B este superior.

**Protocol:** split `split_indices.npz`, 5 seeds, p ∈ {0, 0.001, 0.01, 0.05}, prag max-F1 pe validation la același p, evaluare **doar pe validation** (testul nu se încarcă). Varianta A refolosește ponderile STAR din Exp. 9 după ce reproducerea e verificată; varianta B se reantrenează de la zero (aceeași inițializare și aceeași ordine a mini-batch-urilor pe seed).

In [ ]:
import hashlib
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pennylane as qml
import sklearn
import torch
from IPython.display import display
from scipy import stats
from scipy.stats import spearmanr
from sklearn.metrics import average_precision_score, precision_recall_curve
from sklearn.model_selection import train_test_split

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks").is_dir() and (p / "src").is_dir())
sys.path.insert(0, str(REPO_ROOT))
from src import star_pca as sp
from src.scores_io import check_split

# ---- config (fix, ca în Exp. 9) ----
SPLIT_SEED = 42
SEEDS = [0, 1, 2, 3, 4]
NOISE_LEVELS = [0.001, 0.01, 0.05]
P_ALL = [0.0, *NOISE_LEVELS]
N_QUBITS, EPOCHS, BATCH_SIZE, LEARNING_RATE = sp.N_QUBITS, sp.EPOCHS, sp.BATCH_SIZE, sp.LEARNING_RATE
SHOTS_PER_EXECUTION = 1024    # doar pentru cost ledger (simularea folosește valori de așteptare exacte)
N_BALANCED, N_TRAIN_EXPECTED = 590, 170883
EXPECTED_CNOT, EXPECTED_NOISE_LOCATIONS, EXPECTED_PARAMS = 15, 73, 32

# ---- exploratoriu: ținte de recall ----
RECALL_TARGETS = [0.70, 0.75, 0.80, 0.85]
EXCESS_FP_PRECISION = 0.10    # convenție de proiect: sub această precision marcăm „alarme false excesive"

EXACT_TOL = 1e-12
REPRO_TOL = 1e-9              # reproducerea Exp. 9: istoric de antrenare și scoruri/metrici cu zgomot (ponderile Exp. 9 salvate)
REPRO_W_TOL = 1e-6            # ponderi după reantrenare: diferențe de ultim bit la unghiuri se amplifică peste 570 pași Adam (observat ~2e-9)

VARIANTS = ["balanced", "full_train"]
LABELS = {"balanced": "BALANCED PCA (A)", "full_train": "FULL-TRAIN PCA (B)"}
COLORS = {"balanced": "#2a78d6", "full_train": "#eb6834"}
MARKERS = {"balanced": "o", "full_train": "s"}
SEED_DOT_COLOR = "#8a8f98"
plt.rcParams.update({"axes.grid": True, "grid.alpha": 0.25, "axes.spines.top": False, "axes.spines.right": False})

EXP9_DIR = REPO_ROOT / "results" / "result_exp9"      # doar citit
RESULTS_DIR = REPO_ROOT / "results" / "result_exp10"
WEIGHTS_DIR = RESULTS_DIR / "weights"
PLOTS_DIR = RESULTS_DIR / "plots"
CACHE_DIR = RESULTS_DIR / "scores_cache"              # scoruri pe validation (nu se comit: vezi README-ul rezultatelor)
for d in [PLOTS_DIR, CACHE_DIR, *[WEIGHTS_DIR / v for v in VARIANTS]]:
    d.mkdir(parents=True, exist_ok=True)

torch.set_default_dtype(torch.float64)
sp.set_seed(SPLIT_SEED)
print(f"seeds {SEEDS}   p {P_ALL}   dtype implicit torch: {torch.get_default_dtype()}")
print("pennylane", qml.__version__, "| torch", torch.__version__, "| sklearn", sklearn.__version__, "| seed-ul global este setat (numpy, torch); PennyLane e determinist (default.qubit/default.mixed, fără shots)")

## 1. Date, split și verificări de leakage

In [ ]:
SPLIT_PATH, DATA_PATH = REPO_ROOT / "split_indices.npz", REPO_ROOT / "data" / "creditcard.csv"
split_sha = check_split()   # sha256 = cel din Exp. 5; 170.883 / 56.962 / 56.962 tranzacții; 295 / 99 / 98 fraude
print("sha256(split_indices.npz) =", split_sha, "→ egal cu cel din results/result_exp5/config.json")

df = pd.read_csv(DATA_PATH)
df["Class"] = df["Class"].astype(int)
assert df.shape == (284807, 31) and df.index.equals(pd.RangeIndex(len(df)))
X_all, y_all = df.drop(columns="Class"), df["Class"].to_numpy()
split = np.load(SPLIT_PATH)
train_idx, val_idx, test_idx = split["train"], split["val"], split["test"]

_trval, _test = train_test_split(np.arange(len(df)), test_size=0.2, stratify=y_all, random_state=SPLIT_SEED)
_train, _val = train_test_split(_trval, test_size=0.25, stratify=y_all[_trval], random_state=SPLIT_SEED)
assert np.array_equal(_train, train_idx) and np.array_equal(_val, val_idx) and np.array_equal(_test, test_idx)
assert not (set(train_idx) & set(val_idx)) and not (set(train_idx) & set(test_idx)) and not (set(val_idx) & set(test_idx))

X_train, y_train = X_all.loc[train_idx], y_all[train_idx]
X_val, y_val = X_all.loc[val_idx], y_all[val_idx]
assert (len(train_idx), int(y_train.sum())) == (N_TRAIN_EXPECTED, 295) and (len(val_idx), int(y_val.sum())) == (56962, 99)
assert np.array_equal(X_train.index.to_numpy(), train_idx)
del X_all, y_all, df, _trval, _test, _train, _val, test_idx    # testul nu se folosește în Exp. 10
print(f"train {len(train_idx):,} ({int(y_train.sum())} fraude)   validation {len(val_idx):,} ({int(y_val.sum())} fraude)   test: neatins")

## 2. Cele două encodere și testele lor

`AngleEncoder` (A) este cel din Exp. 1–9, neschimbat. `FullTrainPCAEncoder` (B) schimbă un singur lucru: pe ce date vede `PCA.fit`. Verificăm din cod că scalerele și setul de 590 sunt identice, că atributele `n_samples_seen_` sunt cele așteptate și că unghiurile A reproduc exact hash-ul din Exp. 9.

In [ ]:
exp9_train = {(r["topologie"], r["seed"]): r for r in map(json.loads, (EXP9_DIR / "train_runs.jsonl").read_text(encoding="utf-8").splitlines()) if r["topologie"] == "star"}

seed_data, checks = {}, []
for seed in SEEDS:
    X_s, y_s = sp.undersample(X_train, y_train, seed)
    assert len(y_s) == N_BALANCED and int(y_s.sum()) == 295 and X_s.index.is_unique and set(X_s.index) <= set(X_train.index)

    t0 = time.perf_counter(); enc_A = sp.AngleEncoder(N_QUBITS, seed).fit(X_s); fit_A = time.perf_counter() - t0
    t0 = time.perf_counter(); enc_B = sp.FullTrainPCAEncoder(N_QUBITS, seed).fit(X_s, X_train); fit_B = time.perf_counter() - t0

    # --- populațiile de fit (PCA expune n_samples_, nu n_samples_seen_) ---
    assert enc_A.pca.n_samples_ == N_BALANCED, enc_A.pca.n_samples_
    assert enc_B.pca.n_samples_ == N_TRAIN_EXPECTED, enc_B.pca.n_samples_
    assert enc_A.scaler.n_samples_seen_ == enc_B.scaler.n_samples_seen_ == N_BALANCED
    assert np.array_equal(enc_A.scaler.mean_, enc_B.scaler.mean_) and np.array_equal(enc_A.scaler.scale_, enc_B.scaler.scale_), "StandardScaler diferă"
    assert enc_A.angles.n_samples_seen_ == enc_B.angles.n_samples_seen_ == N_BALANCED
    for enc in (enc_A, enc_B):   # MinMaxScaler fitat exact pe proiecțiile PCA ale celor 590
        proj = enc.pca.transform(enc._scaled(X_s))
        assert np.allclose(enc.angles.data_min_, proj.min(0), atol=EXACT_TOL) and np.allclose(enc.angles.data_max_, proj.max(0), atol=EXACT_TOL)
    # --- intrările circuitului ---
    out = {}
    for v, enc in [("balanced", enc_A), ("full_train", enc_B)]:
        Z_tr, Z_val = enc.transform(X_s), enc.transform(X_val)
        assert Z_tr.shape == (N_BALANCED, N_QUBITS) and Z_val.shape == (len(y_val), N_QUBITS)
        assert np.isfinite(Z_tr).all() and np.isfinite(Z_val).all() and Z_tr.min() >= 0 and Z_val.max() <= np.pi
        out[v] = {"enc": enc, "Z_tr": Z_tr, "Z_val": Z_val,
                  "Z_sha256": hashlib.sha256(Z_tr.tobytes() + Z_val.tobytes()).hexdigest()}
    assert not np.allclose(out["balanced"]["Z_tr"], out["full_train"]["Z_tr"]), "B ar trebui să schimbe unghiurile"
    # unghiurile A = cele din Exp. 9 (același hash salvat)
    # unghiurile A = cele din Exp. 9, verificat numeric: ponderile salvate în Exp. 9 dau pe ele exact AUPRC-ul salvat.
    # (Hash-ul brut al octeților din Exp. 9 NU se reproduce: diferență de ultim bit între sesiuni; verificarea numerică de mai jos e cea relevantă.)
    saved_star = torch.load(EXP9_DIR / "weights" / "star" / f"seed_{seed}.pt", weights_only=True)
    ap_check = average_precision_score(y_val, sp.predict(out["balanced"]["Z_val"], saved_star["weights"]))
    assert abs(ap_check - saved_star["best_val_ap"]) < EXACT_TOL, f"seed {seed}: unghiurile A ≠ Exp. 9 ({ap_check} vs {saved_star['best_val_ap']})"
    same_hash = out["balanced"]["Z_sha256"] == exp9_train[("star", seed)]["sha256 unghiuri (train + val)"]
    seed_data[seed] = {"X_s": X_s, "y_s": y_s, "fit_time": {"balanced": fit_A, "full_train": fit_B}, **out}
    print(f"seed {seed}: PCA_A n={enc_A.pca.n_samples_}   PCA_B n={enc_B.pca.n_samples_}   fit A {fit_A*1e3:.0f} ms, fit B {fit_B*1e3:.0f} ms   unghiuri A: AUPRC Exp. 9 reprodus la {EXACT_TOL:.0e} (hash octeți identic: {same_hash})")

# --- reproductibilitate pe același seed: refit B → aceleași unghiuri, bit cu bit ---
enc_B2 = sp.FullTrainPCAEncoder(N_QUBITS, 0).fit(seed_data[0]["X_s"], X_train)
assert np.array_equal(enc_B2.transform(seed_data[0]["X_s"]), seed_data[0]["full_train"]["Z_tr"]) and np.array_equal(enc_B2.transform(X_val), seed_data[0]["full_train"]["Z_val"])
checks += [("PCA_A.n_samples_ == 590 (5 seeds; sklearn 1.9: PCA nu are n_samples_seen_, doar IncrementalPCA)", True), ("PCA_B.n_samples_ == 170883 (5 seeds)", True),
           ("StandardScaler identic A/B; MinMaxScaler fitat pe aceleași 590 proiectate", True),
           ("unghiurile A: ponderile Exp. 9 dau AUPRC-ul salvat la 1e-12 (hash-ul brut al octeților diferă de Exp. 9)", True), ("refit B pe același seed = aceleași unghiuri (bit cu bit)", True),
           ("train/validation disjuncte; PCA_B vede doar indici din train; testul nu e încărcat", True)]
print("✅ encodere: populațiile de fit, scalerele, MinMaxScaler și reproductibilitatea verificate")

## 3. Circuitul STAR: 15 CNOT, 73 de locuri de zgomot, 32 de parametri

In [ ]:
w_probe = sp.init_weights(0)
x_probe = torch.tensor(seed_data[0]["balanced"]["Z_val"][0])
qnode_ops = qml.transforms.decompose(qml.QNode(sp.star_circuit, qml.device("default.qubit", wires=N_QUBITS)), gate_set=sp.NATIVE_GATES)
ops = qml.workflow.construct_tape(qnode_ops)(x_probe.numpy(), w_probe.detach().numpy()).operations
N_CNOT = sum(op.name == "CNOT" for op in ops)
N_NOISE_LOC = sp.noise_locations(sp.star_circuit, x_probe, w_probe)
N_PARAMS = w_probe.numel()
assert (N_CNOT, N_NOISE_LOC, N_PARAMS) == (EXPECTED_CNOT, EXPECTED_NOISE_LOCATIONS, EXPECTED_PARAMS), (N_CNOT, N_NOISE_LOC, N_PARAMS)
exp9_cfg = json.loads((EXP9_DIR / "config.json").read_text(encoding="utf-8"))
exp9_star_ops = exp9_cfg["topologies"]["star"]["native_op_order"]
assert [f"{op.name}{list(op.wires)}" for op in ops] == exp9_star_ops, "circuitul diferă de STAR din Exp. 9"
EXECUTIONS_PER_SAMPLE = 1 + 2 * N_PARAMS
print(f"✅ STAR: {len(ops)} porți native, {N_CNOT} CNOT, {N_NOISE_LOC} locuri de zgomot, {N_PARAMS} parametri; secvența de porți = Exp. 9 (config.json)")
checks += [("STAR: 15 CNOT, 73 locuri de zgomot, 32 parametri; secvența de porți = Exp. 9", True)]

## 4. Reproducerea referinței STAR (A) și antrenarea lui B

- **A:** reantrenăm STAR cu bucla din `src/star_pca.py` și comparăm cu ponderile salvate în Exp. 9. Dacă nu se reproduce, oprim execuția. Pentru rezultatele lui A folosim ponderile Exp. 9.
- **B:** antrenare de la zero pe cele 590 de exemple transformate de noul encoder, cu aceeași inițializare (`init_weights(seed)`) și aceeași ordine a mini-batch-urilor ca A. Epoca se alege independent, după AUPRC pe validation la p = 0 (regula din Exp. 9).

In [ ]:
def append_jsonl(path, record):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False, default=lambda o: o.item() if isinstance(o, np.generic) else str(o)) + "\n")

models = {v: {} for v in VARIANTS}
repro_rows = []
for seed in SEEDS:
    d = seed_data[seed]
    # ---- A: ponderile Exp. 9 + reantrenare de verificare ----
    saved = torch.load(EXP9_DIR / "weights" / "star" / f"seed_{seed}.pt", weights_only=True)
    assert torch.equal(saved["initial_weights"], sp.init_weights(seed).detach())
    rerun = sp.train_vqc_run(d["balanced"]["Z_tr"], d["y_s"], d["balanced"]["Z_val"], y_val, sp.init_weights(seed), seed)
    w_err = (rerun["weights"] - saved["weights"]).abs().max().item()
    hist_err = max(max(abs(a["train loss"] - b["train loss"]), abs(a["validation AUPRC"] - b["validation AUPRC"])) for a, b in zip(rerun["history"], saved["history"]))
    assert rerun["best_epoch"] == saved["best_epoch"] and w_err < REPRO_W_TOL and hist_err < REPRO_TOL, (seed, rerun["best_epoch"], saved["best_epoch"], w_err, hist_err)
    repro_rows.append({"seed": seed, "epoca aleasă (Exp. 9 / acum)": f"{saved['best_epoch']} / {rerun['best_epoch']}", "max |Δ ponderi|": w_err, "max |Δ istoric|": hist_err,
                       "timp reantrenare [s]": rerun["time_train_s"]})
    exp9_time = exp9_train[("star", seed)]["timp antrenare [s]"]
    models["balanced"][seed] = {"weights": saved["weights"].detach().clone(), "best_epoch": saved["best_epoch"], "best_val_ap": saved["best_val_ap"], "history": saved["history"],
                                "time_train_s": exp9_time, "adam_steps": rerun["adam_steps"], "batch_order": rerun["batch_order_sha256"]}

    # ---- B: antrenare de la zero (cu reluare din fișiere salvate) ----
    path = WEIGHTS_DIR / "full_train" / f"seed_{seed}.pt"
    train_path = RESULTS_DIR / "train_runs.jsonl"
    recorded = {(r["varianta"], r["seed"]): r for r in map(json.loads, train_path.read_text(encoding="utf-8").splitlines())} if train_path.exists() else {}
    if path.exists() and ("full_train", seed) in recorded:
        saved_b, source = torch.load(path, weights_only=True), "încărcat"
        rec = recorded[("full_train", seed)]
        models["full_train"][seed] = {"weights": saved_b["weights"], "best_epoch": saved_b["best_epoch"], "best_val_ap": saved_b["best_val_ap"], "history": saved_b["history"],
                                      "time_train_s": rec["timp antrenare [s]"], "adam_steps": rec["pași Adam"], "batch_order": rec["sha256 ordine mini-batch"]}
        assert torch.equal(saved_b["initial_weights"], sp.init_weights(seed).detach())
    else:
        initial = sp.init_weights(seed)
        run = sp.train_vqc_run(d["full_train"]["Z_tr"], d["y_s"], d["full_train"]["Z_val"], y_val, initial, seed)
        torch.save({"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"], "history": run["history"],
                    "initial_weights": initial.detach().clone()}, path)
        losses = [h["train loss"] for h in run["history"]]
        append_jsonl(train_path, {"varianta": "full_train", "seed": seed, "best_epoch": run["best_epoch"], "val AUPRC (p = 0)": run["best_val_ap"],
                                  "train loss la epoca aleasă": losses[run["best_epoch"] - 1], "train loss final": losses[-1], "epoci executate": len(losses),
                                  "exemple train VQC": len(d["y_s"]), "pași Adam": run["adam_steps"], "timp antrenare [s]": run["time_train_s"],
                                  "sha256 ordine mini-batch": run["batch_order_sha256"], "sha256 unghiuri (train + val)": d["full_train"]["Z_sha256"]})
        models["full_train"][seed] = {"weights": run["weights"], "best_epoch": run["best_epoch"], "best_val_ap": run["best_val_ap"], "history": run["history"],
                                      "time_train_s": run["time_train_s"], "adam_steps": run["adam_steps"], "batch_order": run["batch_order_sha256"]}
        source = f"antrenat și salvat ({run['time_train_s']:.0f} s)"
        if seed == SEEDS[0]:   # determinism: a doua antrenare, același seed, aceleași ponderi
            run2 = sp.train_vqc_run(d["full_train"]["Z_tr"], d["y_s"], d["full_train"]["Z_val"], y_val, sp.init_weights(seed), seed)
            assert torch.equal(run2["weights"], run["weights"]) and run2["best_epoch"] == run["best_epoch"], "antrenarea B nu e deterministă"
            checks.append(("antrenare B, seed 0, de două ori: ponderi identice bit cu bit", True))
    mA, mB = models["balanced"][seed], models["full_train"][seed]
    assert mA["batch_order"] == mB["batch_order"], "ordinea mini-batch-urilor diferă între A și B"
    assert len(mB["history"]) == EPOCHS and mB["best_epoch"] == 1 + int(np.argmax([h["validation AUPRC"] for h in mB["history"]]))
    print(f"seed {seed}: A reprodus (max |Δ ponderi| {w_err:.1e}, epoca {saved['best_epoch']}, val AUPRC p=0 {saved['best_val_ap']:.4f})   B {source}: epoca {mB['best_epoch']:2d}  val AUPRC p=0 {mB['best_val_ap']:.4f}")

repro = pd.DataFrame(repro_rows).set_index("seed")
display(repro)
checks += [("A: reantrenare = ponderile și istoricul Exp. 9 (5 seeds: aceeași epocă, istoric < 1e-9, ponderi < 1e-6)", True),
           ("aceeași inițializare și aceeași ordine a mini-batch-urilor pentru A și B (sha256, 5 seeds)", True),
           (f"VQC antrenat pe exact {N_BALANCED} exemple; tot train-ul intră doar în PCA.fit al lui B", True)]
print("✅ A reprodus din Exp. 9; B antrenat pe aceleași 590 de exemple, aceeași inițializare și aceeași ordine a mini-batch-urilor")

## 5. Evaluare pe validation sub zgomot depolarizant

Modelul oficial de zgomot (`default.mixed`, decompoziție în porți native, `DepolarizingChannel(p)` după fiecare poartă pe fiecare qubit atins), p ∈ {0, 0.001, 0.01, 0.05}. Pragul max-F1 se alege pe validation la același p, adică estimare optimistă. Scorurile se salvează în `results/result_exp10/scores_cache/` ca să se poată relua rularea.

In [ ]:
exp9_val = {(r["seed"], r["p"]): r for r in map(json.loads, (EXP9_DIR / "val_runs.jsonl").read_text(encoding="utf-8").splitlines()) if r["topologie"] == "star"}
scores, eval_time = {}, {}
for seed in SEEDS:
    for v in VARIANTS:
        for p in P_ALL:
            path = CACHE_DIR / f"{v}_seed{seed}_p{p}.npz"
            if path.exists():
                with np.load(path) as z:
                    scores[(v, seed, p)], eval_time[(v, seed, p)] = z["scores"], float(z["time"])
                continue
            Z_val, w = seed_data[seed][v]["Z_val"], models[v][seed]["weights"]
            t0 = time.perf_counter()
            s = sp.predict(Z_val, w) if p == 0.0 else sp.noisy_predict(Z_val, w, p)
            elapsed = time.perf_counter() - t0
            np.savez(path, scores=s, time=elapsed)
            scores[(v, seed, p)], eval_time[(v, seed, p)] = s, elapsed
            print(f"{v:<10} seed {seed} p = {p:<5} AUPRC {average_precision_score(y_val, s):.4f}  ({elapsed:.0f} s)")

# sanity: default.mixed la p = 0 = default.qubit (primele 2000 de tranzacții, ambele variante, seed 0)
for v in VARIANTS:
    mixed0 = sp.noisy_predict(seed_data[0][v]["Z_val"][:2000], models[v][0]["weights"], 0.0)
    assert np.abs(mixed0 - scores[(v, 0, 0.0)][:2000]).max() < 1e-10
checks.append(("default.mixed la p = 0 = default.qubit (2000 tranzacții, A și B)", True))

def run_record(v, seed, p):
    s, s0 = scores[(v, seed, p)], scores[(v, seed, 0.0)]
    threshold = sp.choose_threshold(y_val, s)
    diff = s - s0
    return {"varianta": v, "seed": seed, "p": p, **sp.evaluate(y_val, s, threshold), "threshold": float(threshold),
            "Spearman rho": spearmanr(s0, s).statistic, "MAE scor": np.abs(diff).mean(), "RMSE scor": float(np.sqrt(np.mean(diff ** 2))),
            "CNOT": N_CNOT, "locuri de zgomot": N_NOISE_LOC, "timp antrenare [s]": models[v][seed]["time_train_s"], "timp scorare [s]": eval_time[(v, seed, p)]}

runs = pd.DataFrame([run_record(v, seed, p) for v in VARIANTS for seed in SEEDS for p in P_ALL])
runs["pierdere AUPRC față de p = 0"] = runs.apply(lambda r: runs[(runs.varianta == r.varianta) & (runs.seed == r.seed) & (runs.p == 0.0)]["AUPRC"].iloc[0] - r["AUPRC"], axis=1)
assert len(runs) == 2 * len(SEEDS) * len(P_ALL) and not runs.duplicated(["varianta", "seed", "p"]).any()

# A trebuie să reproducă rezultatele salvate în Exp. 9 (STAR)
err = []
for _, r in runs[runs.varianta == "balanced"].iterrows():
    ref = exp9_val[(r.seed, r.p)]
    err.append(max(abs(r["AUPRC"] - ref["AUPRC"]), abs(r["threshold"] - ref["threshold"]), abs(r["MAE scor"] - ref["MAE scor"]), abs(r["F1"] - ref["F1"]),
                   abs(r["TP"] - ref["TP"]), abs(r["FP"] - ref["FP"]), abs(r["Spearman rho"] - ref["Spearman rho"])))
assert max(err) < REPRO_TOL, f"A nu reproduce STAR din Exp. 9: {max(err)}"
checks.append((f"A sub zgomot = Exp. 9 STAR (AUPRC, prag, TP, FP, F1, MAE, ρ; 20 evaluări; max eroare {max(err):.1e})", True))
print(f"✅ A reproduce Exp. 9 (STAR) la toate cele 20 de (seed, p): eroare max {max(err):.1e}")

ref005 = pd.DataFrame([exp9_val[(s, 0.05)] for s in SEEDS])
print(f"referință Exp. 9 STAR, p = 0.05, 5 seeds: AUPRC {ref005.AUPRC.mean():.3f}, recall {ref005.recall.mean():.4f}, F1 {ref005.F1.mean():.3f}, FP {ref005.FP.mean():.1f}; AUPRC p = 0: {np.mean([exp9_val[(s, 0.0)]['AUPRC'] for s in SEEDS]):.3f}")
runs.to_csv(RESULTS_DIR / "metrics_per_seed.csv", index=False)
display(runs[runs.p == 0.05][["varianta", "seed", "AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "Spearman rho", "MAE scor"]].round(4))

## 6. Rezumat și comparații pereche pe seed

Δ = B − A, calculat pe același seed (aceeași inițializare, același subset de 590, aceeași ordine a mini-batch-urilor). Cu 5 seeds, **nu** declarăm superioritate statistică: raportăm media, std (ddof = 1), intervalul t de 95 % pentru Δ (n = 5), numărul de seeds cu îmbunătățire și testul semnului exact, a cărui valoare minimă posibilă este 0.0625 la 5/5 seeds.

In [ ]:
METRICS = ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP", "FN", "TN", "threshold", "Spearman rho", "MAE scor", "RMSE scor", "pierdere AUPRC față de p = 0"]
LOWER_IS_BETTER = {"FP", "FN", "MAE scor", "RMSE scor", "pierdere AUPRC față de p = 0"}
NEUTRAL = {"threshold", "TN"}                       # fără sens „mai bun/mai rău"
RATE_METRICS = {"precision", "recall", "F1"}

def per_seed(v, p, metric):
    sel = runs[(runs.varianta == v) & (runs.p == p)].set_index("seed").loc[SEEDS]
    return sel[metric].astype(float)

summary = pd.DataFrame([{"varianta": LABELS[v], "p": p, **{f"{m}": per_seed(v, p, m).mean() for m in METRICS}, **{f"{m} std": per_seed(v, p, m).std(ddof=1) for m in METRICS},
                          "timp antrenare [s]": per_seed(v, p, "timp antrenare [s]").mean(), "timp scorare [s]": per_seed(v, p, "timp scorare [s]").mean()}
                         for v in VARIANTS for p in P_ALL])
summary.to_csv(RESULTS_DIR / "summary_metrics.csv", index=False)
fmt = lambda m, p, v, digits=4: f"{per_seed(v, p, m).mean():.{digits}f} ± {per_seed(v, p, m).std(ddof=1):.{digits}f}"
display(pd.DataFrame([{"varianta": LABELS[v], "p": p, **{m: fmt(m, p, v, 1 if m in ("TP", "FP", "FN") else 4) for m in ["AUPRC", "AUROC", "precision", "recall", "F1", "MCC", "TP", "FP", "Spearman rho"]}}
                      for v in VARIANTS for p in P_ALL]).set_index(["varianta", "p"]))

def sign_test_p(d):
    nz = d[d != 0]
    if len(nz) == 0:
        return 1.0
    k = int((nz > 0).sum())
    return float(min(1.0, 2 * stats.binom.cdf(min(k, len(nz) - k), len(nz), 0.5)))

paired_rows = []
for p in P_ALL:
    for m in METRICS:
        a, b = per_seed("balanced", p, m), per_seed("full_train", p, m)
        d = b - a
        better = (d < 0) if m in LOWER_IS_BETTER else (d > 0)
        half = stats.t.ppf(0.975, len(SEEDS) - 1) * d.std(ddof=1) / np.sqrt(len(SEEDS))
        paired_rows.append({"p": p, "metrică": m, "A (BALANCED)": a.mean(), "B (FULL-TRAIN)": b.mean(), "Δ medie (B − A)": d.mean(), "Δ std (ddof=1)": d.std(ddof=1),
                            "Δ medie [puncte procentuale]": 100 * d.mean() if m in RATE_METRICS else np.nan,
                            "IC95% t jos": d.mean() - half, "IC95% t sus": d.mean() + half,
                            "seeds în care B e mai bun": np.nan if m in NEUTRAL else int(better.sum()), "seeds egale": int((d == 0).sum()),
                            "test semn exact (bilateral)": sign_test_p(d), **{f"Δ seed {s}": d[s] for s in SEEDS}})
paired = pd.DataFrame(paired_rows)
paired.to_csv(RESULTS_DIR / "paired_comparison.csv", index=False)
print("Comparație pereche la p = 0.05 (B − A):")
display(paired[paired.p == 0.05].set_index("metrică")[["A (BALANCED)", "B (FULL-TRAIN)", "Δ medie (B − A)", "Δ std (ddof=1)", "Δ medie [puncte procentuale]", "IC95% t jos", "IC95% t sus", "seeds în care B e mai bun", "test semn exact (bilateral)"]].round(4))

## 7. Recall mare: analiză exploratorie a pragului

Regula deterministă: pentru fiecare țintă (70, 75, 80, 85 %), dintre pragurile curbei Precision–Recall cu recall ≥ țintă alegem **cel mai mare** (cele mai puține alarme). **Exploratoriu:** pragul se alege și se evaluează pe același validation. AUPRC nu depinde de prag; coloana „AUPRC" este doar a modelului. Marcăm „alarme false excesive" când precision < 10 %.

In [ ]:
rows = []
for v in VARIANTS:
    for seed in SEEDS:
        for p in P_ALL:
            for target in RECALL_TARGETS:
                r = sp.recall_target_row(y_val, scores[(v, seed, p)], target)
                base = {"varianta": v, "seed": seed, "p": p, "target recall": target}
                if r is None:
                    rows.append({**base, "realizabil": False})
                else:
                    rows.append({**base, "realizabil": True, "threshold": r["threshold"], "recall": r["recall"], "precision": r["precision"], "F1": r["F1"],
                                 "TP": r["TP"], "FP": r["FP"], "FN": r["FN"], "AUPRC": r["AUPRC"], "alarme false excesive (precision < 10%)": bool(r["precision"] < EXCESS_FP_PRECISION)})
recall_analysis = pd.DataFrame(rows)
recall_analysis.to_csv(RESULTS_DIR / "threshold_recall_analysis.csv", index=False)
recall_summary = (recall_analysis.groupby(["p", "target recall", "varianta"])
                  .agg(recall=("recall", "mean"), precision=("precision", "mean"), F1=("F1", "mean"), TP=("TP", "mean"), FP=("FP", "mean"), FN=("FN", "mean"),
                       AUPRC=("AUPRC", "mean"), seeds_cu_alarme_excesive=("alarme false excesive (precision < 10%)", "sum"), realizabil=("realizabil", "sum")))
display(recall_summary.loc[0.05].round(3))
display(recall_summary.loc[0.0].round(3))

## 8. Analiza PCA: ce se schimbă în reprezentare

Varianța explicată **nu** este separabilitate: raportăm separat cât variază datele pe fiecare componentă (`explained_variance_ratio_`) și cât de bine separă fiecare unghi clasele pe validation (AUROC univariat, d Cohen), apoi legătura cu PCA-ul fitat doar pe tranzacțiile legitime din train. Aceste diagnostice folosesc etichetele de validation doar pentru descriere, nu pentru antrenare sau selecție.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score

pca_rows, dim_rows, sim_rows = [], [], []
for seed in SEEDS:
    d = seed_data[seed]
    scaler = d["balanced"]["enc"].scaler
    Xs_full = d["balanced"]["enc"]._scaled(X_train)       # tot train, standardizat cu scalerul celor 590 (identic A/B)
    total_var = Xs_full.var(axis=0, ddof=1).sum()
    legit_pca = PCA(n_components=N_QUBITS, random_state=seed).fit(Xs_full[y_train == 0])
    comps = {v: d[v]["enc"].pca.components_ for v in VARIANTS}
    for v in VARIANTS:
        enc = d[v]["enc"]
        evr = enc.pca.explained_variance_ratio_
        proj_full = (Xs_full - enc.pca.mean_) @ enc.pca.components_.T
        captured_full = proj_full.var(axis=0, ddof=1) / total_var            # varianța din TOT train captată de fiecare componentă a acestui PCA
        pca_rows.append({"varianta": v, "seed": seed, "n_samples_seen": int(enc.pca.n_samples_),
                         **{f"EVR PC{k+1}": evr[k] for k in range(4)}, "EVR cumulat": evr.sum(),
                         **{f"varianță din tot train captată PC{k+1}": captured_full[k] for k in range(4)}, "varianță din tot train captată (suma)": captured_full.sum(),
                         "|cos| subspațiu vs PCA doar-legit (min pe componente)": float(np.abs(np.sum(enc.pca.components_ * legit_pca.components_, axis=1)).min()),
                         "|cos| PC1 vs PCA doar-legit": float(abs(np.dot(enc.pca.components_[0], legit_pca.components_[0])))})
        raw_val = sp.unclipped_angles(enc, X_val)
        raw_tr = sp.unclipped_angles(enc, d["X_s"])
        ang_val = d[v]["Z_val"]
        out_low, out_high = (raw_val < 0), (raw_val > np.pi)
        for k in range(4):
            a_leg, a_fra = ang_val[y_val == 0, k], ang_val[y_val == 1, k]
            auc = roc_auc_score(y_val, ang_val[:, k])
            pooled = np.sqrt((a_leg.var(ddof=1) + a_fra.var(ddof=1)) / 2)
            dim_rows.append({"varianta": v, "seed": seed, "dimensiune": k, "unghi mediu (val)": ang_val[:, k].mean(), "unghi std (val)": ang_val[:, k].std(),
                             "unghi mediu legit": a_leg.mean(), "unghi mediu fraudă": a_fra.mean(), "d Cohen (fraudă − legit)": (a_fra.mean() - a_leg.mean()) / pooled,
                             "AUROC univariat al unghiului": auc, "separare |AUROC − 0.5|·2": abs(auc - 0.5) * 2,
                             "valori val < 0 (înainte de clip)": int(out_low[:, k].sum()), "valori val > π (înainte de clip)": int(out_high[:, k].sum()),
                             "proporție în afara domeniului (val)": float((out_low[:, k] | out_high[:, k]).mean()),
                             "rânduri val cu ≥1 dimensiune în afara domeniului": int((out_low | out_high).any(axis=1).sum()),
                             "valori în afara domeniului (train 590)": int(((raw_tr[:, k] < -1e-12) | (raw_tr[:, k] > np.pi + 1e-12)).sum())})
    cos = np.abs(np.sum(comps["balanced"] * comps["full_train"], axis=1))
    sign = np.sign(np.sum(comps["balanced"] * comps["full_train"], axis=1))
    sim_rows.append({"seed": seed, **{f"|cos| PC{k+1} A vs B": cos[k] for k in range(4)}, **{f"semn PC{k+1} (A·B)": int(sign[k]) for k in range(4)}})

pca_table, dim_table, sim_table = pd.DataFrame(pca_rows), pd.DataFrame(dim_rows), pd.DataFrame(sim_rows)
pca_table.to_csv(RESULTS_DIR / "pca_analysis.csv", index=False)
dim_table.to_csv(RESULTS_DIR / "pca_angle_dimensions.csv", index=False)
sim_table.to_csv(RESULTS_DIR / "pca_component_similarity.csv", index=False)
display(pca_table.groupby("varianta").mean(numeric_only=True).drop(columns="seed").T.round(4))
display(sim_table.set_index("seed").round(3))
display(dim_table.groupby(["varianta", "dimensiune"]).mean(numeric_only=True).drop(columns="seed")[["d Cohen (fraudă − legit)", "AUROC univariat al unghiului", "separare |AUROC − 0.5|·2", "proporție în afara domeniului (val)", "valori val < 0 (înainte de clip)", "valori val > π (înainte de clip)"]].round(4))

## 9. Cost ledger

Execuții **nominale** (parameter shift: 1 + 2·P = 65 execuții/exemplu, 1024 shots ipotetice/execuție) vs. simularea efectivă (valori de așteptare exacte; nu s-au tras shots). Circuitul e același în A și B, deci costul cuantic per execuție (15 CNOT, 73 locuri de zgomot) nu se schimbă. B adaugă doar cost clasic: un `PCA.fit` pe 170.883 de rânduri.

In [ ]:
n_val, n_batches = len(y_val), -(-N_BALANCED // BATCH_SIZE)
assert all(models[v][s]["adam_steps"] == EPOCHS * n_batches for v in VARIANTS for s in SEEDS)
ledger = {"nota": "Execuțiile și shots-urile sunt nominale (estimare pentru hardware). Simularea a folosit valori de așteptare exacte, fără shots. Circuitul STAR este identic în A și B.",
          "circuit": {"CNOT": N_CNOT, "locuri_de_zgomot": N_NOISE_LOC, "parametri": N_PARAMS, "porti_native": len(ops), "executii_per_exemplu_parameter_shift": EXECUTIONS_PER_SAMPLE},
          "shots_ipotetice_per_executie": SHOTS_PER_EXECUTION, "seeds": SEEDS, "variante": {}}
for v in VARIANTS:
    per_seed_exec = {"antrenare (forward + parameter shift, nominal)": EPOCHS * N_BALANCED * EXECUTIONS_PER_SAMPLE,
                     "validare după fiecare epocă (p = 0)": EPOCHS * n_val,
                     "validation p = 0, ponderile alese": n_val,
                     "validation sub zgomot (p = 0.001, 0.01, 0.05)": len(NOISE_LEVELS) * n_val}
    total = sum(per_seed_exec.values())
    train_t = [models[v][s]["time_train_s"] for s in SEEDS]
    eval_t = [sum(eval_time[(v, s, p)] for p in P_ALL) for s in SEEDS]
    ledger["variante"][v] = {
        "executii_per_seed": per_seed_exec, "executii_total_per_seed": total, "executii_total_5_seeds": total * len(SEEDS),
        "shots_ipotetice_total_5_seeds": total * len(SEEDS) * SHOTS_PER_EXECUTION,
        "pasi_Adam_per_seed": EPOCHS * n_batches, "pasi_Adam_total_5_seeds": EPOCHS * n_batches * len(SEEDS),
        "evaluari_validation_per_seed": {"in_timpul_antrenarii (o data pe epoca, p = 0)": EPOCHS, "finale (p = 0 si 3 niveluri de zgomot)": len(P_ALL)},
        "timp_simulare_antrenare_s": {"per_seed": train_t, "total": float(np.sum(train_t))},
        "timp_simulare_evaluare_validation_s": {"per_seed": eval_t, "total": float(np.sum(eval_t))},
        "cost_clasic_PCA_fit_s": {"per_seed": [seed_data[s]["fit_time"][v] for s in SEEDS], "total": float(sum(seed_data[s]["fit_time"][v] for s in SEEDS))},
        "randuri_folosite_la_PCA_fit": N_BALANCED if v == "balanced" else N_TRAIN_EXPECTED,
        "nota_timp_antrenare": "A: timpul din Exp. 9 (altă sesiune); B: măsurat acum. Nu sunt comparabile exact."}
(RESULTS_DIR / "cost_ledger.json").write_text(json.dumps(ledger, indent=2, ensure_ascii=False), encoding="utf-8")
display(pd.DataFrame({LABELS[v]: {"execuții/seed": ledger["variante"][v]["executii_total_per_seed"], "execuții 5 seeds": ledger["variante"][v]["executii_total_5_seeds"],
                                   "shots ipotetice 5 seeds": ledger["variante"][v]["shots_ipotetice_total_5_seeds"], "pași Adam 5 seeds": ledger["variante"][v]["pasi_Adam_total_5_seeds"],
                                   "simulare antrenare [s]": ledger["variante"][v]["timp_simulare_antrenare_s"]["total"], "simulare evaluare [s]": ledger["variante"][v]["timp_simulare_evaluare_validation_s"]["total"],
                                   "PCA.fit clasic [s]": ledger["variante"][v]["cost_clasic_PCA_fit_s"]["total"]} for v in VARIANTS}).map(lambda x: f"{x:,.3f}" if isinstance(x, float) else f"{x:,}"))

## 10. Grafice

In [ ]:
xpos = np.arange(len(P_ALL))
xlabels_p = ["0", *[str(p) for p in NOISE_LEVELS]]
OFFSETS = {"balanced": -0.06, "full_train": 0.06}

def plot_vs_p(metric, ylabel, title, filename, legend_loc="best"):
    fig, ax = plt.subplots(figsize=(7.5, 4.4))
    for v in VARIANTS:
        vals = [per_seed(v, p, metric) for p in P_ALL]
        for i, x in enumerate(vals):
            ax.scatter(np.full(len(x), xpos[i] + OFFSETS[v]), x, s=12, color=SEED_DOT_COLOR, alpha=0.6, zorder=2)
        ax.errorbar(xpos + OFFSETS[v], [x.mean() for x in vals], yerr=[x.std(ddof=1) for x in vals], color=COLORS[v], marker=MARKERS[v], markersize=8,
                    linewidth=2, capsize=4, zorder=3, label=LABELS[v])
    ax.set_xticks(xpos, xlabels_p); ax.set_xlabel("probabilitate depolarizantă p per poartă"); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.legend(fontsize=9, loc=legend_loc); plt.tight_layout(); fig.savefig(PLOTS_DIR / filename, dpi=130); plt.show()

plot_vs_p("AUPRC", "AUPRC (validation)", "STAR ZZ · AUPRC vs p · medie ± std pe 5 seeds (gri: seeds)", "1_auprc_vs_p.png", "lower left")
plot_vs_p("recall", "Recall la pragul max-F1 (același p)", "STAR ZZ · Recall vs p · prag F1 optim · medie ± std", "2_recall_vs_p.png", "lower left")
plot_vs_p("F1", "F1 la pragul max-F1 (același p)", "STAR ZZ · F1 vs p · medie ± std · optimist (prag pe validation)", "3_f1_vs_p.png", "lower left")

def paired_plot(metric, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(5.6, 4.4))
    a, b = per_seed("balanced", 0.05, metric), per_seed("full_train", 0.05, metric)
    for s in SEEDS:
        ax.plot([0, 1], [a[s], b[s]], color=SEED_DOT_COLOR, alpha=0.8, linewidth=1.3, marker="o", markersize=4, zorder=2)
        ax.annotate(f"seed {s}", (1.02, b[s]), fontsize=8, color="0.35", va="center")
    for i, (v, x) in enumerate([("balanced", a), ("full_train", b)]):
        ax.errorbar([i], [x.mean()], yerr=[x.std(ddof=1)], color=COLORS[v], marker=MARKERS[v], markersize=10, capsize=5, linewidth=2, zorder=3)
    ax.set_xticks([0, 1], [LABELS["balanced"], LABELS["full_train"]]); ax.set_xlim(-0.3, 1.45); ax.set_ylabel(ylabel); ax.set_title(title)
    plt.tight_layout(); fig.savefig(PLOTS_DIR / filename, dpi=130); plt.show()

paired_plot("AUPRC", "AUPRC (validation)", "AUPRC la p = 0.05, pe seed (linie = același seed)", "4_auprc_p005_paired.png")
paired_plot("recall", "Recall (prag max-F1 la p = 0.05)", "Recall la p = 0.05, pe seed", "5_recall_p005_per_seed.png")

# 6. curbele Precision–Recall la p = 0.05
fig, axes = plt.subplots(1, 3, figsize=(15, 4.6), sharex=True, sharey=True)
seed_colors = plt.cm.viridis(np.linspace(0.05, 0.9, len(SEEDS)))
for ax, v in zip(axes[:2], VARIANTS):
    for s, c in zip(SEEDS, seed_colors):
        pr, rc, _ = precision_recall_curve(y_val, scores[(v, s, 0.05)])
        ax.plot(rc, pr, color=c, linewidth=1.4, label=f"seed {s} (AUPRC {average_precision_score(y_val, scores[(v, s, 0.05)]):.3f})")
        r = runs[(runs.varianta == v) & (runs.seed == s) & (runs.p == 0.05)].iloc[0]
        ax.scatter([r.recall], [r.precision], color=c, s=40, edgecolor="black", zorder=3)
    ax.set_title(LABELS[v] + " · ● = prag max-F1"); ax.set_xlabel("recall"); ax.legend(fontsize=8, loc="lower left")
axes[0].set_ylabel("precision")
for v in VARIANTS:
    for s in SEEDS:
        pr, rc, _ = precision_recall_curve(y_val, scores[(v, s, 0.05)])
        axes[2].plot(rc, pr, color=COLORS[v], alpha=0.65, linewidth=1.2, label=LABELS[v] if s == SEEDS[0] else None)
axes[2].set_title("suprapunere (5 seeds fiecare)"); axes[2].set_xlabel("recall"); axes[2].legend(fontsize=9)
fig.suptitle("Curba Precision–Recall la p = 0.05 (validation)", fontsize=13); plt.tight_layout(); fig.savefig(PLOTS_DIR / "6_pr_curves_p005.png", dpi=130); plt.show()

# 7. distribuția celor 4 unghiuri (seed 0)
fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), sharey=False)
for k, ax in enumerate(axes):
    for v in VARIANTS:
        Z = seed_data[0][v]["Z_val"]
        for cls, ls, name in [(0, "-", "legit"), (1, "--", "fraudă")]:
            ax.hist(Z[y_val == cls, k], bins=40, range=(0, np.pi), density=True, histtype="step", color=COLORS[v], linestyle=ls, linewidth=1.6,
                    label=f"{LABELS[v].split(' (')[0]} · {name}")
    ax.set_title(f"unghi qubit {k}"); ax.set_xlabel("unghi [rad]"); ax.set_yscale("log")
axes[0].set_ylabel("densitate (scară log)"); axes[0].legend(fontsize=7, loc="upper left")
fig.suptitle("Distribuția unghiurilor cuantice pe validation (seed 0, după clip la [0, π])", fontsize=13); plt.tight_layout(); fig.savefig(PLOTS_DIR / "7_angle_distributions.png", dpi=130); plt.show()

# 8. recall vs alarme false (curba PR transformată), p = 0.05
fig, ax = plt.subplots(figsize=(7.5, 4.8))
for v in VARIANTS:
    for s in SEEDS:
        pr, rc, _ = precision_recall_curve(y_val, scores[(v, s, 0.05)])
        tp = rc * int(y_val.sum()); fp = np.where(pr > 0, tp / np.clip(pr, 1e-12, None) - tp, 0)
        ax.plot(np.clip(fp, 0.5, None), rc, color=COLORS[v], alpha=0.6, linewidth=1.2, label=LABELS[v] if s == SEEDS[0] else None)
        r = runs[(runs.varianta == v) & (runs.seed == s) & (runs.p == 0.05)].iloc[0]
        ax.scatter([max(r.FP, 0.5)], [r.recall], color=COLORS[v], marker=MARKERS[v], s=45, edgecolor="black", zorder=3)
for t in RECALL_TARGETS:
    ax.axhline(t, color="0.6", linestyle=":", linewidth=1)
ax.set_xscale("log"); ax.set_xlabel("alarme false (FP, scară log)"); ax.set_ylabel("recall"); ax.legend(fontsize=9, loc="lower right")
ax.set_title("Recall vs alarme false, p = 0.05\nmarcaje = prag max-F1 · linii punctate = ținte 70/75/80/85 %", fontsize=11)
plt.tight_layout(); fig.savefig(PLOTS_DIR / "8_recall_vs_fp_p005.png", dpi=130); plt.show()

# 9. varianța explicată pe componente
fig, ax = plt.subplots(figsize=(6.4, 4.2))
w = 0.38
for i, v in enumerate(VARIANTS):
    sub = pca_table[pca_table.varianta == v]
    m = [sub[f"EVR PC{k+1}"].mean() for k in range(4)]; sd = [sub[f"EVR PC{k+1}"].std(ddof=1) for k in range(4)]
    ax.bar(np.arange(4) + (i - 0.5) * w, m, w, yerr=sd, color=COLORS[v], capsize=3, label=LABELS[v])
ax.set_xticks(range(4), [f"PC{k+1}" for k in range(4)]); ax.set_ylabel("explained_variance_ratio_ (medie ± std pe 5 seeds)")
ax.set_title("Varianța explicată (≠ separabilitate)"); ax.legend(fontsize=9); plt.tight_layout(); fig.savefig(PLOTS_DIR / "9_explained_variance.png", dpi=130); plt.show()

## 11. Config și verificări salvate

In [ ]:
config = {
    "experiment": "balanced_vs_full_train_pca_star_zz",
    "architecture": "STAR ZZ from Exp. 9 (pairs (0,1),(0,2),(0,3); feature map x1, ansatz x3; 4 qubits; 32 params; readout PauliZ(0); score (1-expval)/2)",
    "variant_A": "AngleEncoder: StandardScaler(Time, Amount) + PCA(4) + MinMaxScaler(0, pi), all fitted on the 590 balanced examples; transform clips to [0, pi]",
    "variant_B": "same StandardScaler (590) ; PCA(4).fit on all 170,883 train rows standardised with that scaler ; MinMaxScaler fitted on the B-PCA projections of the same 590 ; same clipping",
    "vqc_training_set": "590 balanced examples (undersample(X_train, y_train, seed)); the full train set is used only for PCA.fit of variant B",
    "A_weights": "results/result_exp9/weights/star (reproduced by retraining, not retrained for the results)", "B_weights": "retrained from scratch (results/result_exp10/weights/full_train)",
    "seeds": SEEDS, "noise_levels": P_ALL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "optimizer": "Adam", "loss": "BCELoss",
    "init": "init_weights(seed), identical for A and B", "shuffle": "torch.Generator().manual_seed(seed), identical for A and B (sha256 verified)",
    "epoch_selection": "best validation AUPRC at p = 0 (same rule for A and B)",
    "noise_model": "default.mixed, decomposition into NATIVE_GATES, DepolarizingChannel(p) after every gate on every wire it touches",
    "threshold": "max F1 on validation scores at the same p (selected and evaluated on the same set: optimistic)",
    "recall_targets_exploratory": {"targets": RECALL_TARGETS, "rule": "highest PR-curve threshold with recall >= target", "excess_fp_flag": f"precision < {EXCESS_FP_PRECISION}"},
    "evaluation_set": "validation only (test not loaded)", "split_sha256": split_sha, "split": "split_indices.npz (verified against official reconstruction and Exp. 5 hash)",
    "tolerances": {"exact": EXACT_TOL, "reproduction_vs_exp9": REPRO_TOL}, "shots_per_execution_hypothetical": SHOTS_PER_EXECUTION,
    "versions": {"pennylane": qml.__version__, "torch": torch.__version__, "sklearn": sklearn.__version__, "numpy": np.__version__},
    "code": "src/star_pca.py", "checks": [c[0] for c in checks],
}
(RESULTS_DIR / "config.json").write_text(json.dumps(config, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
pd.DataFrame(checks, columns=["verificare", "trecut"]).to_csv(RESULTS_DIR / "checks.csv", index=False)
repro.to_csv(RESULTS_DIR / "exp9_reproduction.csv")
display(pd.DataFrame(checks, columns=["verificare", "trecut"]))
print("salvat în", RESULTS_DIR.relative_to(REPO_ROOT))